In [ ]:
import cProfile
import json
import time

# Part (a): Inverted Index
def create_index(corpus):
    index = {}
    for doc_id, document in enumerate(corpus):
        for term in document.split():
            term = term.lower()  # Ensure case-insensitivity
            if term not in index:
                index[term] = [doc_id]
            elif doc_id not in index[term]:
                index[term].append(doc_id)
    return index

# Part (b): Boolean Search
def boolean_search(query, index):
    def find_documents(term):
        return index.get(term.lower(), [])

    def intersect(list1, list2):
        return list(set(list1) & set(list2))

    def union(list1, list2):
        return list(set(list1) | set(list2))

    query_terms = query.split()
    stack = []

    for term in query_terms:
        if term == '&':
            if len(stack) < 2:
                print("Skipping '&' due to insufficient operands")
                continue  # Skip this operation if there aren't enough operands
            right = stack.pop()
            left = stack.pop()
            stack.append(intersect(left, right))
        elif term:
            current_docs = find_documents(term)
            if stack and isinstance(stack[-1], list):
                previous_docs = stack.pop()
                stack.append(union(previous_docs, current_docs))
            else:
                stack.append(current_docs)

    if len(stack) != 1:
        print("Error in query processing; final stack should have exactly one item")
        return []
    return stack[0]


def load_dataset(file_path):
    corpus = []
    with open(file_path, 'r', encoding='utf-8') as file:
        for i, line in enumerate(file):
            try:
                doc = json.loads(line)
                corpus.append(doc['short_description'])
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")
            except KeyError:
                print(f"Key 'short_description' not found in document: {i+1}")



    return corpus






# Main function to run experiments
def run_experiments(file_path, queries):
    # Load the dataset and create the index
    corpus = load_dataset(file_path)
    index = create_index(corpus)
    
    # Measure the time taken to create the index
    index_creation_start_time = time.time()
    index_creation_time = (time.time() - index_creation_start_time) * 10**6  # Convert to microseconds
    print(f"Time taken to create index: {index_creation_time} microseconds")
    
    # Process each query and measure the time taken for search
    for query in queries:
        # Ensure spaces around '&' to separate terms correctly
        formatted_query = ' '.join(query.split()).replace("&", " & ").replace("  ", " ")
        search_start_time = time.time()
        
        try:
            results = boolean_search(formatted_query, index)
            search_time = (time.time() - search_start_time) * 10**6  # Convert to microseconds
            
            # Output results
            if results:
                print(f"Query: {formatted_query}")
                print(f"Number of results: {len(results)}")
                print(f"Time taken for search: {search_time} microseconds\n")
            else:
                print(f"Query: {formatted_query}")
                print("Number of results: 0")
                print(f"Time taken for search: {search_time} microseconds\n")
        
        except Exception as e:
            print(f"An error occurred while processing query '{formatted_query}': {e}")
            print(f"Time since search start: {(time.time() - search_start_time) * 10**6} microseconds\n")


# Example usage
if __name__ == "__main__":
    file_path = r'data/input.json'
    queries = ["apple", "samsung sony&tv", "samsung&apple", "samsung apple"]
    cProfile.run('run_experiments(file_path, queries)')